# Demo - Integration Gate Against a Real Diff
**Day 1 - Session 1, Topic 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/demos-notebook/demo-integration-gate-real-diff.ipynb)

**Goal:** Admit or block two real commits by comparing each task card's declared file ownership with the actual `git diff --name-only` of that commit.

The blocked case is produced, not described. One simulated agent stays inside its card and one also edits the shared router. **Both commits pass their focused tests**, so this shows the gate catching a boundary violation that a green test suite cannot see.

> Requires Git. Runs in a temporary repository with no API key.


## 1. Setup

Locate the course files and put `demo_support` on the import path.


In [ ]:
# Setup: make the course files and demo_support importable.
# On Colab nothing is present yet, so clone the companion repo once.
# Locally this finds your existing checkout and clones nothing.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
MARKER = Path("lab-workspace-solution") / "router.py"


def find_repo_root():
    directory = Path.cwd()
    for _ in range(6):
        if (directory / MARKER).exists():
            return directory
        directory = directory.parent
    clone = Path.cwd() / "agent-orchestration-companion"
    if not (clone / MARKER).exists():
        print(f"Cloning {{REPO_URL}} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
            check=True,
            env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
        )
    return clone


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "day1" / "demos"))

# Colab has no global Git identity; the demos set a local one per sandbox repo.
print("Course root:", ROOT)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

## 2. Read ownership from the real task cards

Allowed files come from `task-cards/*.md` on disk, not from a literal in this notebook.


In [ ]:
import re
from pathlib import Path

from demo_support import assert_true, git, heading, run_tests, sandbox, show_evidence

OWNED_PATTERN = re.compile(r"^-\s+`([^`]+)`", flags=re.MULTILINE)

sandbox_context = sandbox()
WORK, BASE = sandbox_context.__enter__()
show_evidence("approved base commit", BASE)


def declared_ownership(root, card_name):
    """Parse the owned-files list and acceptance command out of a real task card."""
    card = (root / "task-cards" / card_name).read_text()
    owned = OWNED_PATTERN.findall(card)
    command = re.search(r"```bash\n(.+?)\n```", card, flags=re.DOTALL)
    return set(owned), (command.group(1).strip() if command else "")


for card in ("email.md", "sms.md"):
    owned, command = declared_ownership(WORK, card)
    heading(f"task-cards/{card}")
    show_evidence("owned files", ", ".join(sorted(owned)))
    show_evidence("acceptance command", command)

## 3. Create two real commits from two simulated sessions

The email agent stays in bounds. The SMS agent also edits the shared router.


In [ ]:
SMS_MAX = "MAX_LENGTH = 160"

git("checkout", "-q", "-b", "agent/email", BASE, cwd=WORK)
email_path = WORK / "channels" / "email.py"
email_path.write_text(email_path.read_text().replace("your order shipped", "your order has shipped"))
git("add", "-A", cwd=WORK)
git("commit", "-q", "-m", "Refine email wording", cwd=WORK)
EMAIL_COMMIT = git("rev-parse", "--short", "HEAD", cwd=WORK)
git("checkout", "-q", "main", cwd=WORK)

git("checkout", "-q", "-b", "agent/sms", BASE, cwd=WORK)
sms_path = WORK / "channels" / "sms.py"
sms_path.write_text(sms_path.read_text().replace(SMS_MAX, "MAX_LENGTH = 140"))
router_path = WORK / "router.py"
router_path.write_text(router_path.read_text().replace(
    'CHANNEL_ORDER = ("email", "sms")', 'CHANNEL_ORDER = ("sms", "email")'
))
git("add", "-A", cwd=WORK)
git("commit", "-q", "-m", "Shorten SMS and reorder channels", cwd=WORK)
SMS_COMMIT = git("rev-parse", "--short", "HEAD", cwd=WORK)
git("checkout", "-q", "main", cwd=WORK)

heading("Two real commits from two simulated agent sessions")
show_evidence("agent/email commit", EMAIL_COMMIT)
show_evidence("agent/sms commit", SMS_COMMIT)

## 4. Gate each commit on its real diff

Git reports the changed paths. The card declares what was allowed. Compare them.


In [ ]:
def gate(root, base, commit, card_name):
    owned, command = declared_ownership(root, card_name)
    output = git("diff", "--name-only", f"{base}..{commit}", cwd=root)
    actual = set(output.splitlines()) if output else set()
    git("checkout", "-q", commit, cwd=root)  # test the commit under review
    passed, _ = run_tests(command.replace("python3 -m unittest ", ""), root) if command else (False, "")
    git("checkout", "-q", "main", cwd=root)
    return {"owned": owned, "actual": actual, "outside": actual - owned, "tests_passed": passed}


def report(name, result):
    heading(f"Gate: {name}")
    show_evidence("card allows", ", ".join(sorted(result["owned"])))
    show_evidence("git diff changed", ", ".join(sorted(result["actual"])))
    show_evidence("focused tests", "passed" if result["tests_passed"] else "FAILED")
    if result["outside"]:
        show_evidence("outside ownership", ", ".join(sorted(result["outside"])))
    decision = "ACCEPT" if result["tests_passed"] and not result["outside"] else "BLOCK"
    show_evidence("decision", decision)
    if decision == "BLOCK":
        reason = ("tests are green but the diff left the card's boundary"
                  if result["tests_passed"] else "the focused acceptance command did not pass")
        show_evidence("reason", reason)
    return decision


EMAIL_RESULT = gate(WORK, BASE, EMAIL_COMMIT, "email.md")
EMAIL_DECISION = report("agent/email", EMAIL_RESULT)

SMS_RESULT = gate(WORK, BASE, SMS_COMMIT, "sms.md")
SMS_DECISION = report("agent/sms", SMS_RESULT)

## 5. Verify the evidence

The key check: the blocked commit's own tests passed, so tests alone would have admitted it.


In [ ]:
heading("Evidence checks")
assert_true(EMAIL_DECISION == "ACCEPT", "the compliant commit was admitted")
assert_true(SMS_DECISION == "BLOCK", "the overreaching commit was blocked")
assert_true(SMS_RESULT["tests_passed"],
            "the blocked commit's own tests passed, so tests alone would have admitted it")
assert_true("router.py" in SMS_RESULT["outside"],
            "Git reported the out-of-bounds file, it was not hardcoded")

sandbox_context.__exit__(None, None, None)
print("\nTakeaway: Gate on the real diff against declared ownership,")
print("because a passing test cannot detect a boundary violation.")

### Expected output

- Both task cards print their real owned-files list and acceptance command.
- Two real short commit SHAs appear (they differ on every run).
- `agent/email`: diff shows only `channels/email.py`, tests pass, **ACCEPT**.
- `agent/sms`: diff shows `channels/sms.py` **and** `router.py`, tests still
  pass, yet the decision is **BLOCK** with `router.py` outside ownership.
- Four `[verified]` lines, then the takeaway.
